# AuraGateway CUDA 12.9 dynamic-linker inspection v1

Bounded diagnostic inspection only. This notebook does not install packages, load a model, issue requests, or claim qualification.


In [ ]:

from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import time
import zipfile
from datetime import UTC, datetime
from pathlib import Path, PurePosixPath
from typing import Any

NOTEBOOK_NAME = "auragateway-cu129-dynlink-inspect-v1"
INPUT_DIRECTORY_NAME = "auragateway_vllm_cu129_wheelhouse_v1"
OUTPUT_DIRECTORY_NAME = "auragateway_cu129_dynlink_inspection_evidence_v1"
OUTPUT_ZIP = Path(f"/kaggle/working/{OUTPUT_DIRECTORY_NAME}.zip")
EVIDENCE_ROOT = Path(f"/kaggle/working/{OUTPUT_DIRECTORY_NAME}")
EXTRACT_ROOT = Path("/kaggle/working/auragateway_cu129_dynlink_inspection_extract_v1")
MAX_EXCERPT = 30000

SOURCE_V4_EVIDENCE_ZIP_SHA256 = (
    "9a4bd10a66c440ffb2628f98ce143e38a1b5cdb06a745497b7b386910816e0fe"
)
EXPECTED_CONTROL_HASHES = {
    "resolution_lock.json": (
        "1575538b0a412c9b030fc95ccada0f0527553b76f06ef6b2b72904e61c84870c"
    ),
    "sha256_manifest.json": (
        "789fb23ab7d9c4f28dd909e808a53a65d692c0d7b43bc44da9e974817d771b8d"
    ),
    "materialization_receipt.json": (
        "52aa42b940dd606ab5685686ab893eb085efed2a7466989f654e870f4b360589"
    ),
}
SELECTED_PACKAGES = (
    "nvidia-cusparse-cu12",
    "nvidia-nvjitlink-cu12",
    "nvidia-cuda-runtime-cu12",
)
REQUIRED_SYMBOL = "__nvJitLinkGetErrorLogSize_12_9"
RELEVANT_LINE_PATTERN = re.compile(
    r"(nvjitlink|cusparse|cudart|search path|trying file|version lookup|undefined symbol)",
    re.IGNORECASE,
)


def canonical_json(payload: object) -> str:
    return json.dumps(payload, ensure_ascii=True, separators=(",", ":"), sort_keys=True)


def streaming_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sanitize(text: str | bytes | None) -> str:
    if text is None:
        return ""
    decoded = text.decode("utf-8", errors="replace") if isinstance(text, bytes) else text
    replacements = {
        "/kaggle/input": "<input>",
        "/kaggle/working": "<working>",
        os.environ.get("HOME", ""): "<home>",
    }
    for source, replacement in replacements.items():
        if source:
            decoded = decoded.replace(source, replacement)
    return decoded[-MAX_EXCERPT:]


def filtered_excerpt(text: str | bytes | None) -> str:
    if text is None:
        return ""
    decoded = text.decode("utf-8", errors="replace") if isinstance(text, bytes) else text
    relevant = [line for line in decoded.splitlines() if RELEVANT_LINE_PATTERN.search(line)]
    return sanitize("\n".join(relevant))


def write_record(name: str, payload: object) -> None:
    (EVIDENCE_ROOT / name).write_text(
        canonical_json(payload) + "\n",
        encoding="utf-8",
    )


def run_command(
    role: str,
    argv: list[str],
    *,
    timeout: float,
    environment: dict[str, str] | None = None,
    filter_output: bool = False,
) -> dict[str, object]:
    started = time.monotonic()
    started_at = datetime.now(UTC).isoformat(timespec="seconds")
    env = {**os.environ} if environment is None else environment
    try:
        result = subprocess.run(
            argv,
            check=False,
            capture_output=True,
            text=True,
            timeout=timeout,
            env=env,
        )
        excerpt = filtered_excerpt if filter_output else sanitize
        return {
            "schema_version": "1.0.0",
            "command_role": role,
            "argv": [sanitize(item) for item in argv],
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": result.returncode,
            "timed_out": False,
            "status": "PASSED" if result.returncode == 0 else "FAILED",
            "stdout_excerpt": excerpt(result.stdout),
            "stderr_excerpt": excerpt(result.stderr),
        }
    except subprocess.TimeoutExpired as exc:
        excerpt = filtered_excerpt if filter_output else sanitize
        return {
            "schema_version": "1.0.0",
            "command_role": role,
            "argv": [sanitize(item) for item in argv],
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": None,
            "timed_out": True,
            "status": "FAILED",
            "stdout_excerpt": excerpt(exc.stdout),
            "stderr_excerpt": excerpt(exc.stderr),
        }


def normalize_name(value: str) -> str:
    return re.sub(r"[-_.]+", "-", value).lower()


def require_json_object(path: Path) -> dict[str, Any]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError(f"expected one JSON object: {path.name}")
    return payload


def validate_input(input_root: Path) -> tuple[dict[str, object], dict[str, dict[str, Any]]]:
    if not input_root.is_dir() or input_root.is_symlink():
        raise RuntimeError("governed wheelhouse root is missing or unsafe")
    for relative, expected in EXPECTED_CONTROL_HASHES.items():
        path = input_root / relative
        if not path.is_file() or path.is_symlink():
            raise RuntimeError(f"control file is missing or unsafe: {relative}")
        if streaming_sha256(path) != expected:
            raise RuntimeError(f"control file identity drifted: {relative}")

    resolution = require_json_object(input_root / "resolution_lock.json")
    records = resolution.get("records")
    if (
        resolution.get("package_count") != 176
        or resolution.get("review_decision") != "APPROVED_AS_EXACT_LOCKED_CLOSURE"
        or not isinstance(records, list)
        or len(records) != 176
    ):
        raise RuntimeError("resolution lock contract drifted")

    selected: dict[str, dict[str, Any]] = {}
    for record in records:
        if not isinstance(record, dict):
            raise RuntimeError("resolution-lock record is invalid")
        normalized = record.get("normalized_name")
        filename = record.get("artifact_filename")
        digest = record.get("sha256")
        version = record.get("version")
        if not all(isinstance(value, str) for value in (normalized, filename, digest, version)):
            raise RuntimeError("resolution-lock record fields are invalid")
        key = normalize_name(normalized)
        if key not in SELECTED_PACKAGES:
            continue
        wheel = input_root / "wheels" / filename
        if not wheel.is_file() or wheel.is_symlink():
            raise RuntimeError(f"selected wheel is missing or unsafe: {filename}")
        if streaming_sha256(wheel) != digest:
            raise RuntimeError(f"selected wheel identity drifted: {filename}")
        selected[key] = {
            "normalized_name": key,
            "version": version,
            "artifact_filename": filename,
            "sha256": digest,
            "size_bytes": wheel.stat().st_size,
            "path": wheel,
        }

    if set(selected) != set(SELECTED_PACKAGES):
        missing = sorted(set(SELECTED_PACKAGES) - set(selected))
        raise RuntimeError("selected package records missing: " + ", ".join(missing))

    return (
        {
            "schema_version": "1.0.0",
            "status": "PASSED",
            "input_directory_name": INPUT_DIRECTORY_NAME,
            "source_v4_evidence_zip_sha256": SOURCE_V4_EVIDENCE_ZIP_SHA256,
            "resolution_lock_sha256": EXPECTED_CONTROL_HASHES["resolution_lock.json"],
            "sha256_manifest_sha256": EXPECTED_CONTROL_HASHES["sha256_manifest.json"],
            "materialization_receipt_sha256": EXPECTED_CONTROL_HASHES[
                "materialization_receipt.json"
            ],
            "selected_packages": list(SELECTED_PACKAGES),
            "network_access_requested": False,
            "model_requests_performed": 0,
            "qualification_claimed": False,
        },
        selected,
    )


def safe_extract_wheel(wheel: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(wheel) as archive:
        for info in archive.infolist():
            relative = PurePosixPath(info.filename)
            if relative.is_absolute() or ".." in relative.parts:
                raise RuntimeError(f"unsafe wheel member: {info.filename}")
            mode = info.external_attr >> 16
            if stat.S_ISLNK(mode):
                raise RuntimeError(f"wheel symlink member is prohibited: {info.filename}")
            target = destination.joinpath(*relative.parts)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open("wb") as sink:
                shutil.copyfileobj(source, sink)


def unique_library(root: Path, pattern: str) -> Path:
    candidates = tuple(path for path in root.rglob(pattern) if path.is_file())
    if len(candidates) != 1:
        raise RuntimeError(
            f"expected exactly one {pattern} below {root.name}; observed {len(candidates)}"
        )
    return candidates[0]


def bounded_candidate_roots() -> tuple[Path, ...]:
    roots = [
        EXTRACT_ROOT,
        Path("/usr/lib/x86_64-linux-gnu"),
        Path("/usr/local/lib"),
        Path("/usr/local/lib/python3.12/dist-packages/nvidia"),
        Path("/opt/conda/lib"),
        Path("/opt/conda/lib/python3.12/site-packages/nvidia"),
    ]
    roots.extend(sorted(Path("/usr/local").glob("cuda*")))
    return tuple(root for root in roots if root.exists())


def discover_candidates(name: str) -> list[Path]:
    observed: dict[str, Path] = {}
    for root in bounded_candidate_roots():
        try:
            matches = root.rglob(name)
            for path in matches:
                if not path.is_file():
                    continue
                key = str(path.resolve())
                observed.setdefault(key, path)
        except PermissionError:
            continue
    return [observed[key] for key in sorted(observed)]


def library_record(path: Path) -> dict[str, object]:
    return {
        "path": sanitize(str(path)),
        "realpath": sanitize(str(path.resolve())),
        "sha256": streaming_sha256(path),
        "size_bytes": path.stat().st_size,
    }


def readelf_symbols(path: Path) -> tuple[dict[str, object], bool]:
    record = run_command(
        "readelf_nvjitlink_symbols",
        ["readelf", "-Ws", str(path)],
        timeout=120.0,
        filter_output=True,
    )
    combined = str(record["stdout_excerpt"]) + "\n" + str(record["stderr_excerpt"])
    return record, REQUIRED_SYMBOL in combined


def parse_ldd_resolution(record: dict[str, object], library: str) -> str | None:
    combined = str(record.get("stdout_excerpt", "")) + "\n" + str(
        record.get("stderr_excerpt", "")
    )
    pattern = re.compile(
        rf"{re.escape(library)}\s*=>\s*(?P<path>\S+)",
        re.IGNORECASE,
    )
    match = pattern.search(combined)
    if match is None:
        return None
    return sanitize(match.group("path"))


def relevant_environment() -> dict[str, object]:
    names = (
        "LD_LIBRARY_PATH",
        "PYTHONPATH",
        "PYTHONHOME",
        "PYTHONNOUSERSITE",
        "CUDA_HOME",
        "CUDA_PATH",
        "CONDA_PREFIX",
        "VIRTUAL_ENV",
        "NVIDIA_VISIBLE_DEVICES",
    )
    return {
        "schema_version": "1.0.0",
        "python_executable": sanitize(sys.executable),
        "python_version": sys.version.split()[0],
        "sys_path": [sanitize(item) for item in sys.path],
        "variables": {name: sanitize(os.environ.get(name, "")) for name in names},
        "network_access_requested": False,
        "model_requests_performed": 0,
        "qualification_claimed": False,
    }


def main() -> int:
    for path in (EVIDENCE_ROOT, EXTRACT_ROOT):
        if path.exists():
            shutil.rmtree(path)
    if OUTPUT_ZIP.exists():
        OUTPUT_ZIP.unlink()
    EVIDENCE_ROOT.mkdir(parents=True)

    summary: dict[str, object]
    try:
        matches = tuple(Path("/kaggle/input").rglob(INPUT_DIRECTORY_NAME))
        if len(matches) != 1:
            raise RuntimeError("expected exactly one governed wheelhouse directory")
        input_root = matches[0]
        input_validation, selected = validate_input(input_root)
        write_record("00_input_validation.json", input_validation)
        write_record("10_environment.json", relevant_environment())

        wheel_selection = {
            "schema_version": "1.0.0",
            "packages": [
                {
                    key: sanitize(str(value)) if key == "path" else value
                    for key, value in selected[name].items()
                }
                for name in SELECTED_PACKAGES
            ],
        }
        write_record("20_wheel_selection.json", wheel_selection)

        extracted_roots: dict[str, Path] = {}
        for package in SELECTED_PACKAGES:
            destination = EXTRACT_ROOT / package
            safe_extract_wheel(Path(selected[package]["path"]), destination)
            extracted_roots[package] = destination

        cusparse = unique_library(extracted_roots["nvidia-cusparse-cu12"], "libcusparse.so.12")
        nvjitlink = unique_library(
            extracted_roots["nvidia-nvjitlink-cu12"], "libnvJitLink.so.12"
        )
        cudart_candidates = tuple(
            path
            for path in extracted_roots["nvidia-cuda-runtime-cu12"].rglob("libcudart.so*")
            if path.is_file()
        )
        if not cudart_candidates:
            raise RuntimeError("no libcudart library found in selected runtime wheel")
        cudart = sorted(cudart_candidates, key=lambda path: len(path.name))[0]

        libraries = {
            "libcusparse": library_record(cusparse),
            "libnvJitLink": library_record(nvjitlink),
            "libcudart": library_record(cudart),
        }
        write_record(
            "30_extracted_library_inventory.json",
            {"schema_version": "1.0.0", "libraries": libraries},
        )

        dynamic = run_command(
            "readelf_cusparse_dynamic",
            ["readelf", "-d", str(cusparse)],
            timeout=120.0,
            filter_output=True,
        )
        versions = run_command(
            "readelf_cusparse_versions",
            ["readelf", "--version-info", str(cusparse)],
            timeout=120.0,
            filter_output=True,
        )
        symbol_record, extracted_symbol_present = readelf_symbols(nvjitlink)
        write_record(
            "40_symbol_and_dynamic_contract.json",
            {
                "schema_version": "1.0.0",
                "required_symbol": REQUIRED_SYMBOL,
                "required_symbol_present_in_extracted_nvjitlink": extracted_symbol_present,
                "cusparse_dynamic": dynamic,
                "cusparse_versions": versions,
                "nvjitlink_symbols": symbol_record,
            },
        )

        candidates = discover_candidates("libnvJitLink.so.12")
        candidate_records = []
        for candidate in candidates:
            symbol_probe, symbol_present = readelf_symbols(candidate)
            candidate_records.append(
                {
                    **library_record(candidate),
                    "required_symbol_present": symbol_present,
                    "symbol_probe_status": symbol_probe["status"],
                }
            )
        write_record(
            "50_nvjitlink_candidates.json",
            {
                "schema_version": "1.0.0",
                "required_symbol": REQUIRED_SYMBOL,
                "candidates": candidate_records,
            },
        )

        inherited_env = {**os.environ, "PYTHONNOUSERSITE": "1"}
        target_dirs = [
            str(nvjitlink.parent),
            str(cusparse.parent),
            str(cudart.parent),
        ]
        existing_ld = inherited_env.get("LD_LIBRARY_PATH", "")
        target_first_env = {
            **inherited_env,
            "LD_LIBRARY_PATH": os.pathsep.join(
                target_dirs + ([existing_ld] if existing_ld else [])
            ),
        }

        inherited_ldd = run_command(
            "inherited_environment_ldd",
            ["ldd", str(cusparse)],
            timeout=120.0,
            environment=inherited_env,
            filter_output=True,
        )
        target_ldd = run_command(
            "target_first_environment_ldd",
            ["ldd", str(cusparse)],
            timeout=120.0,
            environment=target_first_env,
            filter_output=True,
        )
        write_record("60_inherited_environment_ldd.json", inherited_ldd)
        write_record("61_target_first_environment_ldd.json", target_ldd)

        load_code = (
            "import ctypes,sys;"
            "ctypes.CDLL(sys.argv[1],mode=ctypes.RTLD_GLOBAL);"
            "print('loaded')"
        )
        inherited_debug_env = {**inherited_env, "LD_DEBUG": "libs,versions"}
        target_debug_env = {**target_first_env, "LD_DEBUG": "libs,versions"}
        inherited_load = run_command(
            "inherited_environment_cusparse_load",
            [sys.executable, "-c", load_code, str(cusparse)],
            timeout=120.0,
            environment=inherited_debug_env,
            filter_output=True,
        )
        target_load = run_command(
            "target_first_environment_cusparse_load",
            [sys.executable, "-c", load_code, str(cusparse)],
            timeout=120.0,
            environment=target_debug_env,
            filter_output=True,
        )
        write_record("70_inherited_environment_cusparse_load.json", inherited_load)
        write_record("71_target_first_environment_cusparse_load.json", target_load)

        inherited_resolved = parse_ldd_resolution(inherited_ldd, "libnvJitLink.so.12")
        target_resolved = parse_ldd_resolution(target_ldd, "libnvJitLink.so.12")
        inherited_passed = inherited_load["status"] == "PASSED"
        target_passed = target_load["status"] == "PASSED"

        if not extracted_symbol_present:
            assignment = "GOVERNED_NVJITLINK_SYMBOL_MISSING"
        elif not inherited_passed and target_passed:
            assignment = "LOADER_PRECEDENCE_CONFIRMED"
        elif not inherited_passed and not target_passed:
            assignment = "TARGET_FIRST_LOAD_STILL_FAILS"
        elif inherited_passed:
            assignment = "DIRECT_CUSPARSE_LOAD_PASSES_UNDER_INHERITED_ENVIRONMENT"
        else:
            assignment = "INCONCLUSIVE"

        summary = {
            "schema_version": "1.0.0",
            "diagnostic_id": NOTEBOOK_NAME,
            "captured_at": datetime.now(UTC).isoformat(timespec="seconds"),
            "inspection_status": "COMPLETED",
            "root_cause_assignment": assignment,
            "required_symbol": REQUIRED_SYMBOL,
            "required_symbol_present_in_extracted_nvjitlink": extracted_symbol_present,
            "inherited_environment_load_status": inherited_load["status"],
            "target_first_environment_load_status": target_load["status"],
            "inherited_nvjitlink_resolution": inherited_resolved,
            "target_first_nvjitlink_resolution": target_resolved,
            "selected_package_versions": {
                name: selected[name]["version"] for name in SELECTED_PACKAGES
            },
            "source_v4_evidence_zip_sha256": SOURCE_V4_EVIDENCE_ZIP_SHA256,
            "network_access_requested": False,
            "package_installation_performed": False,
            "model_loaded": False,
            "model_requests_performed": 0,
            "benchmark_trajectory_requests_performed": 0,
            "qualification_claimed": False,
            "credentials_used": False,
            "customer_data_used": False,
        }
    except Exception as exc:
        summary = {
            "schema_version": "1.0.0",
            "diagnostic_id": NOTEBOOK_NAME,
            "captured_at": datetime.now(UTC).isoformat(timespec="seconds"),
            "inspection_status": "FAILED_TO_COMPLETE",
            "root_cause_assignment": "INCONCLUSIVE",
            "error_type": type(exc).__name__,
            "error_message": sanitize(str(exc)),
            "source_v4_evidence_zip_sha256": SOURCE_V4_EVIDENCE_ZIP_SHA256,
            "network_access_requested": False,
            "package_installation_performed": False,
            "model_loaded": False,
            "model_requests_performed": 0,
            "benchmark_trajectory_requests_performed": 0,
            "qualification_claimed": False,
            "credentials_used": False,
            "customer_data_used": False,
        }

    write_record("90_summary.json", summary)
    payloads = tuple(sorted(EVIDENCE_ROOT.iterdir(), key=lambda item: item.name))
    manifest = {
        "schema_version": "1.0.0",
        "entries": [
            {
                "path": path.name,
                "sha256": streaming_sha256(path),
                "size_bytes": path.stat().st_size,
            }
            for path in payloads
        ],
    }
    write_record("99_evidence_sha256.json", manifest)

    with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(EVIDENCE_ROOT.iterdir(), key=lambda item: item.name):
            archive.write(path, arcname=path.name)

    print(f"artifact={OUTPUT_ZIP}")
    print(f"size_bytes={OUTPUT_ZIP.stat().st_size}")
    print(f"sha256={streaming_sha256(OUTPUT_ZIP)}")
    print(f"inspection_status={summary['inspection_status']}")
    print(f"root_cause_assignment={summary['root_cause_assignment']}")
    print(
        "inherited_environment_load_status="
        + str(summary.get("inherited_environment_load_status"))
    )
    print(
        "target_first_environment_load_status="
        + str(summary.get("target_first_environment_load_status"))
    )
    print("package_installation_performed=false")
    print("model_requests_performed=0")
    print("qualification_claimed=false")
    print("upload_only_this_file=true")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
